
# VadCLIP — Khuếch Đại Tín Hiệu Lớp Yếu, **Train Từ Đầu** — Bản Kaggle

Khuếch đại gradient về bốn lớp yếu nhất, nhưng **huấn luyện lại từ đầu chỉ với trọng số
CLIP**, không nạp `model_ucf.pth`.

## ⚠️ Đọc trước: con số 88,19 không phải mục tiêu của notebook này

88,19 sinh ra dưới một công thức khác hẳn:

| | Lần chạy cho 88,19 | Notebook này |
|---|---|---|
| Khởi tạo | **Nạp `model_ucf.pth`** (đã ở 88,02) | **Từ đầu, chỉ trọng số CLIP** |
| Số epoch | 1 | 10 |
| Learning rate | 2e-6 | 2e-5 |
| Lịch giảm lr | MultiStepLR([2]) | MultiStepLR([4, 8]) |
| Cắt gradient | 1,0 | tắt |

Lần đó là **tinh chỉnh giai đoạn 2**: lấy một mô hình đã hội tụ ở 88,02 rồi đẩy nó lên
88,19 bằng một epoch với learning rate cực nhỏ. Phần lớn con số 88,19 là công của
`model_ucf.pth`, không phải của phép khuếch đại — phép khuếch đại đóng góp **+0,17** so
với chính điểm xuất phát của nó, và **+0,38** so với đối chứng cùng luật chọn.

Chạy đúng công thức đó mà không nạp trọng số cũ thì ra rác: 1 epoch ở lr 2e-6 từ khởi tạo
ngẫu nhiên gần như không học được gì. Nên notebook này dùng **lịch huấn luyện từ đầu của
baseline** (10 epoch, lr 2e-5, milestones [4, 8], không cắt gradient) — giống hệt lịch mà
`baseline_ctrl` của hướng shift-consistency đã dùng.

**Vậy con số nào là thước đo?** Không phải một con số tuyệt đối, mà là **hiệu giữa
`class_mu12_scratch` và `ctrl_scratch`** — cùng lịch, cùng seed, cùng luật chọn trọng số,
chỉ khác đúng một biến là μ. Đó mới là công của phương pháp. Để tham chiếu: các lần chạy
từ đầu trong dự án này rơi vào khoảng 85,8–88,1 tuỳ seed và luật chọn.

## Phương pháp đang làm gì

Gradient chảy về bốn lớp yếu nhất được **nhân lên μ = 12 lần**. Bốn lớp đó là
`Explosion`, `RoadAccidents`, `Shooting`, `Shoplifting` — bốn lớp AUC thấp nhất mà vẫn còn
≥ 20 video test để con số theo lớp có nghĩa. `Abuse` cũng kém nhưng chỉ có 2 video test
nên bị loại có chủ ý.

`--rescale-mode class` nghĩa là chỉ **gradient** về logits nhánh A của các lớp đích bị
khuếch đại; giá trị loss in ra và nhánh C không bị đụng. Khác với `video`, vốn nhân cả
loss của toàn video đích.

Vì train từ đầu nên **không có θ' để neo về**, do đó `--regularizer` bắt buộc là `none`.
Script từ chối chạy nếu bạn bật EWC/L2 cùng lúc với `--use-pretrained-model false`.

## Bốn lần chạy, và vì sao phải đủ bốn

| Tag | mode | μ | seed |
|---|---|---|---|
| `ctrl_scratch` | `off` | 1,0 | 234 |
| `class_mu12_scratch` | `class` | **12,0** | 234 |
| `ctrl_scratch_s1234` | `off` | 1,0 | 1234 |
| `class_mu12_scratch_s1234` | `class` | **12,0** | 1234 |

Trọng số được giữ là **cực đại của 130 lần chấm trên tập kiểm tra**. Cách chọn đó làm con
số bị thiên lệch lên, và mức thiên lệch tỉ lệ với độ nhiễu. Nên đối chứng **bắt buộc phải
chạy dưới đúng luật chọn ấy** — áp luật chọn đỉnh cho lần có can thiệp mà không áp cho
đối chứng thì mọi chênh lệch đều vô nghĩa.

Hai seed vì ở giai đoạn 2, seed 1234 cho kết quả yếu hơn hẳn seed 234. Một seed không nói
lên điều gì.

## Chấm điểm bao nhiêu lần mỗi epoch

Giữ nguyên nhịp của bản Colab. `--eval-steps 1280`, và trong `ucf_train_rescale.py` biến
đếm là `step = i * batch_size * 2 = i * 128`. Điều kiện chấm là
`step % 1280 == 0 and step != 0`, tức **chấm mỗi 10 bước tối ưu**.

Một epoch có 125 bước, nên:

```
12 lần giữa epoch  (bước 10, 20, ..., 120)
+ 1 lần cuối epoch
= 13 lần mỗi epoch  ->  130 lần cho cả 10 epoch
```

Trong CSV, cột `step` ghi `global_step`, nên trong epoch đầu 13 lần đó hiện ra là
11, 21, 31, …, 121, 125.

## ⚠️ Bốn lần chạy KHÔNG vừa một phiên Kaggle

Mỗi lần chạy giờ là 1.250 bước cộng 130 lần chấm điểm trên 290 video test, chứ không phải
125 bước cộng 13 lần chấm như bản 1 epoch. Phiên Kaggle tối đa **12 giờ** và quota **30
giờ/tuần**, nên hãy tính chạy khoảng **một cặp mỗi phiên**.

Notebook **bỏ qua lần chạy đã có trọng số trong `/kaggle/working/models`**, nên bật
Persistence rồi chạy lại là nó đi tiếp chứ không làm lại. Muốn ép chỉ chạy vài tag thì
sửa `ONLY_TAGS` ở mục 1.

Thứ tự chạy đặt cặp seed 234 trước, vì đó là cặp cho con số chính.

## Chuẩn bị trước khi mở notebook

### Code

Mục 1 tự clone `vngclinh/Finetune-VadCLIP`, hoặc dùng Dataset code nếu có. Cần **cả hai**
thư mục `VadCLIP/src` và `VadCLIP/src_rescale_ewc`: `_bootstrap.py` của thư mục thứ hai
nạp `model.py`, `clip/` và `utils/` từ thư mục thứ nhất, thay vì copy để hai cây khỏi
trôi khỏi nhau.

> ⚠️ **Phải push code trước.** Cờ `--use-pretrained-model` vừa được thêm vào
> `ucf_option_rescale.py`. Chưa push thì mục 4 báo "bản cũ" và dừng.

### `model_ucf.pth` — nay là TUỲ CHỌN

Notebook này không nạp nó để huấn luyện. Có nó thì mục 8 chấm thêm một dòng `source` làm
mốc đối chiếu (88,02); không có cũng chạy bình thường.

### Feature

Dataset công khai **`beosngu/ucf-crime-vadclip-features`** (14 GB): panel bên phải →
**Add Input** → **Datasets** → dán đường dẫn đó → **Add**.

### Session

GPU T4 hoặc P100, Internet **On**, Persistence **Files + Variables** (bắt buộc, để chạy
nhiều phiên).



## 1. Cấu Hình

Tự dò dataset trong `/kaggle/input`. Dò sai thì điền tay vào các biến `*_OVERRIDE`.

Code phải nằm ở nơi ghi được vì `ucf_train_rescale.py` ghi `model/` theo thư mục hiện
hành, mà `/kaggle/input` chỉ đọc. Cell này copy `src/`, `src_rescale_ewc/` và `list/` sang
`/kaggle/working`, giữ nguyên cấu trúc cạnh nhau để `_bootstrap.py` tìm được `../src`.


In [ ]:

from pathlib import Path
import os
import shutil
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
WORK       = Path('/kaggle/working')
TEMP       = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')

CODE_SOURCE   = 'auto'       # 'auto' | 'github' | 'dataset'
GITHUB_REPO   = 'https://github.com/vngclinh/Finetune-VadCLIP.git'
GITHUB_BRANCH = 'main'
FEATURE_DATASET_HINT = 'beosngu/ucf-crime-vadclip-features'   # 14 GB, công khai

# Điền tay nếu tự dò sai. Ví dụ: Path('/kaggle/input/vadclip-rescale')
CODE_OVERRIDE    = None
FEATURE_OVERRIDE = None
CKPT_OVERRIDE    = None


def walk_dirs(root, maxdepth=8):
    """Duyệt thư mục theo bề rộng, CÓ đi xuyên symlink.

    Không dùng rglob: `**` của pathlib gọi is_dir(follow_symlinks=False), tức nó cố ý
    bỏ qua thư mục symlink. Kaggle mount dataset bằng symlink, nên rglob không bao giờ
    nhìn thấy gì bên trong /kaggle/input.
    """
    if not root.exists():
        return
    seen, queue = set(), [(root, 0)]
    while queue:
        directory, depth = queue.pop(0)
        try:
            key = directory.resolve()
        except OSError:
            key = directory
        if key in seen:
            continue
        seen.add(key)
        yield directory
        if depth >= maxdepth:
            continue
        try:
            queue.extend((child, depth + 1)
                         for child in sorted(directory.iterdir()) if child.is_dir())
        except (PermissionError, OSError):
            pass


def find_in_input(*markers, maxdepth=8):
    for directory in walk_dirs(INPUT_ROOT, maxdepth):
        if all((directory / m).exists() for m in markers):
            return directory
    return None


def clone_repo():
    clone_dir = WORK / 'repo'
    if (clone_dir / '.git').exists():
        print('Đã có repo, cập nhật về bản mới nhất ...')
        subprocess.run(['git', '-C', str(clone_dir), 'fetch', '--depth', '1',
                        'origin', GITHUB_BRANCH], check=True)
        subprocess.run(['git', '-C', str(clone_dir), 'reset', '--hard',
                        f'origin/{GITHUB_BRANCH}'], check=True)
    else:
        print('Clone', GITHUB_REPO, '...')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH,
                        GITHUB_REPO, str(clone_dir)], check=True)
    sha = subprocess.run(['git', '-C', str(clone_dir), 'rev-parse', '--short', 'HEAD'],
                         capture_output=True, text=True).stdout.strip()
    print('Commit:', sha)
    return clone_dir / 'VadCLIP'


CODE_FROM_GITHUB = False
CODE_ROOT = CODE_OVERRIDE
if CODE_ROOT is None and CODE_SOURCE != 'github':
    # Phải có CẢ HAI cây: src_rescale_ewc nạp model.py/clip/utils từ src.
    CODE_ROOT = find_in_input('src_rescale_ewc/ucf_train_rescale.py', 'src/model.py')
if CODE_ROOT is None and CODE_SOURCE in ('auto', 'github'):
    CODE_ROOT = clone_repo()
    CODE_FROM_GITHUB = True


def find_feature_root():
    for directory in walk_dirs(INPUT_ROOT):
        if (directory / 'Abuse').is_dir() and (directory / 'Vandalism').is_dir():
            return directory, 'thấy Abuse + Vandalism'
    for directory in walk_dirs(INPUT_ROOT):
        if (directory / 'Abuse').is_dir():
            return directory, 'chỉ thấy Abuse'
    for directory in walk_dirs(INPUT_ROOT):
        try:
            children = [d for d in directory.iterdir() if d.is_dir()]
        except (PermissionError, OSError):
            continue
        with_npy = [d for d in children if next(d.glob('*.npy'), None) is not None]
        if len(with_npy) >= 10:
            return directory, f'{len(with_npy)} thư mục con có file .npy'
    return None, None


def print_input_tree(levels=6):
    print()
    print('=' * 78)
    print('/kaggle/input đang có gì:')
    if not INPUT_ROOT.exists() or not any(INPUT_ROOT.iterdir()):
        print('   (trống — chưa Add Input dataset nào)')
        return
    def walk(path, depth):
        if depth > levels:
            return
        try:
            children = sorted(path.iterdir())
        except (PermissionError, OSError):
            return
        for child in children[:15]:
            npy = len(list(child.glob('*.npy'))) if child.is_dir() else 0
            mark = f'  <- {npy} file .npy' if npy else ''
            print('   ' + '   ' * depth + child.name + ('/' if child.is_dir() else '') + mark)
            if child.is_dir() and not npy:
                walk(child, depth + 1)
        if len(children) > 15:
            print('   ' + '   ' * depth + f'... còn {len(children) - 15} mục nữa')
    walk(INPUT_ROOT, 0)
    print('=' * 78)


if FEATURE_OVERRIDE is not None:
    FEATURE_ROOT, how = FEATURE_OVERRIDE, 'FEATURE_OVERRIDE đặt tay'
else:
    FEATURE_ROOT, how = find_feature_root()

if FEATURE_ROOT is None:
    print('CHƯA DÒ RA DATASET FEATURE.')
    print('Panel bên phải -> Add Input -> Datasets -> dán:', FEATURE_DATASET_HINT)
    print('Đã add rồi mà vẫn báo thế này thì Run -> Restart & Clear Cell Outputs.')
    print_input_tree()
else:
    print('Feature dò ra bằng:', how)

CKPT_ROOT = CKPT_OVERRIDE or find_in_input('model_ucf.pth')
STAGE1_MODEL = (CKPT_ROOT / 'model_ucf.pth') if CKPT_ROOT else None

# --- Code sang nơi ghi được -------------------------------------------------------
PROJECT     = WORK / 'vadclip'
SRC_DIR     = PROJECT / 'src'                 # _bootstrap.py tìm đúng thư mục này
RESCALE_DIR = PROJECT / 'src_rescale_ewc'
LIST_DIR    = PROJECT / 'list'
for name, destination in (('src', SRC_DIR), ('src_rescale_ewc', RESCALE_DIR),
                          ('list', LIST_DIR)):
    if not destination.exists():
        print('Copy', name, '->', destination)
        shutil.copytree(CODE_ROOT / name, destination)
# __pycache__ đi theo từ dataset sẽ che mất file .py mới. Xoá cho chắc.
for cache in PROJECT.rglob('__pycache__'):
    shutil.rmtree(cache, ignore_errors=True)

# --- Nơi ghi kết quả --------------------------------------------------------------
# Giữ lại (vào Output): trọng số cuối của 4 lần chạy, log, CSV.
# Vứt đi (sang /kaggle/temp): model_cur, checkpoint chọn-theo-AUC, epoch checkpoint.
RESULT_DIR = WORK / 'results'
LOG_DIR    = RESULT_DIR / 'logs'
MODEL_DIR  = WORK / 'models'
SCRATCH    = TEMP / 'rescale_scratch'
for directory in (RESULT_DIR, LOG_DIR, MODEL_DIR, SCRATCH):
    directory.mkdir(parents=True, exist_ok=True)

# Tên riêng: KHÔNG dùng lại rescale_1epoch_*.csv của giai đoạn 2. Trộn hai bộ số
# vào một file là cách nhanh nhất để sau này đọc nhầm cấu hình.
METRICS_CSV  = str(RESULT_DIR / 'rescale_scratch_metrics.csv')
PERCLASS_CSV = RESULT_DIR / 'rescale_scratch_perclass.csv'

TRAIN_LIST = str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')
TEST_LIST  = str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv')
GT_ARGS = [
    '--gt-path',         str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path',   str(LIST_DIR / 'gt_label_ucf.npy'),
]

# ================= CẤU HÌNH =================
# Phần KHUẾCH ĐẠI: chép nguyên từ train_rescale_1epoch_colab.ipynb.
TARGET_CLASSES = ['Explosion', 'RoadAccidents', 'Shooting', 'Shoplifting']
MU             = 12.0            # hệ số khuếch đại
RESCALE_MODE   = 'class'         # chỉ khuếch đại gradient về logits nhánh A của lớp đích
EVAL_STEPS     = 1280            # -> chấm mỗi 10 bước -> 13 lần/epoch -> 130 lần cả run
SELECT_METRIC  = 'classifier_auc'   # giữ trọng số có AUC tổng thể cao nhất
BATCH_SIZE     = 64              # KHÔNG hạ: lô nhỏ có thể toàn mẫu đích -> gradient nổ
NUM_WORKERS    = 4

# Phần KHỞI TẠO VÀ LỊCH HUẤN LUYỆN: KHÔNG phải của bản Colab.
# Bản Colab tinh chỉnh 1 epoch ở lr 2e-6 TỪ model_ucf.pth. Ở đây train từ đầu, nên
# dùng lịch từ-đầu của baseline — đúng lịch mà baseline_ctrl hướng shift-consistency
# đã dùng. Chạy 1 epoch ở 2e-6 từ khởi tạo ngẫu nhiên thì không học được gì.
USE_PRETRAINED       = False     # <-- chỉ dùng trọng số CLIP, không nạp model_ucf.pth
MAX_EPOCH            = 10
LR                   = '2e-5'
SCHEDULER_MILESTONES = [4, 8]
SCHEDULER_RATE       = 0.1
GRAD_CLIP            = 0.0       # 0 = tắt, như baseline; stage 2 dùng 1.0

# Train từ đầu thì không có theta' để neo về, nên bắt buộc tắt hợp nhất trọng số.
REGULARIZER = 'none'
LAMBDA_REG  = 0.0
LAMBDA_AUTO = 0.0

# Bốn lần chạy: (tag, rescale_mode, mu, seed). Cặp seed 234 đặt trước vì đó là cặp
# cho con số chính, và bốn lần chạy không vừa một phiên Kaggle.
RUNS = [
    ('ctrl_scratch',             'off',   1.0, 234),
    ('class_mu12_scratch',       'class', 12.0, 234),
    ('ctrl_scratch_s1234',       'off',   1.0, 1234),
    ('class_mu12_scratch_s1234', 'class', 12.0, 1234),
]
# Để trống = chạy tất cả (bỏ qua cái đã có trọng số). Điền tag để chỉ chạy vài cái.
ONLY_TAGS = []
# ============================================

# Cả hai cây phải nằm trên sys.path của CHÍNH notebook, không chỉ của subprocess:
# preflight ở mục 4 import utils.layers, mà utils/ nằm trong src/, không phải
# src_rescale_ewc/. Các lệnh train/eval chạy bằng subprocess thì _bootstrap.py tự lo,
# nhưng nó không chạy trong kernel này.
#
# RESCALE_DIR chèn đầu, SRC_DIR nối cuối — cùng thứ tự mà _bootstrap.py dùng, để
# module của src_rescale_ewc luôn thắng khi trùng tên.
sys.path.insert(0, str(RESCALE_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))
os.chdir(RESCALE_DIR)
PY = [sys.executable, '-u']

print()
print('Nguồn code    :', 'GitHub (' + GITHUB_BRANCH + ')' if CODE_FROM_GITHUB
      else 'Dataset ' + str(CODE_ROOT))
print('src           :', SRC_DIR)
print('src_rescale   :', RESCALE_DIR)
print('list          :', LIST_DIR)
print('Feature       :', FEATURE_ROOT)
print('Stage-1 model :', STAGE1_MODEL, '| có:', bool(STAGE1_MODEL and STAGE1_MODEL.exists()))
print('Kết quả       :', RESULT_DIR)
print('File tạm      :', SCRATCH)
print()
print('Khởi tạo      :', 'TỪ ĐẦU, chỉ trọng số CLIP' if not USE_PRETRAINED
      else f'nạp {STAGE1_MODEL}')
print('Khuếch đại    : mu =', MU, '| mode =', RESCALE_MODE,
      '| lớp đích:', ', '.join(TARGET_CLASSES))
print('Lịch          :', MAX_EPOCH, 'epoch | lr', LR,
      '| milestones', SCHEDULER_MILESTONES, '| grad_clip', GRAD_CLIP)
print('Chấm điểm     : eval_steps', EVAL_STEPS, '-> 13 lần/epoch ->',
      13 * MAX_EPOCH, 'lần cả run | chọn theo', SELECT_METRIC)
selected = [r[0] for r in RUNS if not ONLY_TAGS or r[0] in ONLY_TAGS]
print('Sẽ chạy       :', selected)
if not USE_PRETRAINED:
    print()
    print('LƯU Ý: đây KHÔNG phải cấu hình đã cho 88,19. Con số đó là tinh chỉnh 1 epoch')
    print('       từ model_ucf.pth (vốn đã ở 88,02). Thước đo ở đây là HIỆU giữa')
    print('       class_mu12_scratch và ctrl_scratch cùng seed.')


## 2. Dependencies

In [ ]:
!pip -q install ftfy regex


## 3. Nạp Sẵn Trọng Số CLIP

Không bắt buộc. Có file `ViT-B-16.pt` trong Dataset thì đỡ tải 335 MB mỗi phiên.


In [ ]:

CLIP_SHA256 = '5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f'
cache_dir = Path.home() / '.cache' / 'clip'
cache_dir.mkdir(parents=True, exist_ok=True)
target = cache_dir / 'ViT-B-16.pt'

source = next((d / 'ViT-B-16.pt' for d in walk_dirs(INPUT_ROOT)
               if (d / 'ViT-B-16.pt').exists()), None)
if target.exists():
    print('Đã có sẵn trong cache:', target)
elif source is None:
    print('Không thấy ViT-B-16.pt trong /kaggle/input. CLIP sẽ tự tải (cần Internet: On).')
else:
    print('Copy', source, '->', target)
    shutil.copy2(source, target)

if target.exists():
    import hashlib
    print('SHA256 khớp:', hashlib.sha256(target.read_bytes()).hexdigest() == CLIP_SHA256)



## 4. Preflight

Kiểm mọi thứ trước khi tiêu giờ GPU. Bốn nhóm:

1. Có GPU không.
2. **Có `model_ucf.pth` không** — giai đoạn 2 tinh chỉnh *từ* mô hình đã hội tụ, không có
   nó thì không có gì để tinh chỉnh. Đây là lỗi dừng, không phải cảnh báo.
3. Đủ file code và file danh sách chưa; `ucf_option_rescale.py` có đủ bộ cờ chưa.
4. `utils/layers.py` đã vá chưa — bản upstream ghi cứng `.to('cuda')` trong
   `DistanceAdj.forward`. Kiểm bằng cách chạy thật, không phải bằng cách đọc mã.


In [ ]:

import csv
import importlib

import numpy as np
import torch

_preflight_done = False
GT_MISSING = []


def preflight(force=False):
    global _preflight_done, GT_MISSING
    if _preflight_done and not force:
        return True

    problems = []

    if not torch.cuda.is_available():
        problems.append('Không có GPU. Settings -> Accelerator -> GPU T4 x2 hoặc P100.')

    if FEATURE_ROOT is None:
        print_input_tree()
        problems.append('Không dò ra dataset feature. Đặt FEATURE_OVERRIDE ở mục 1.')

    have_stage1 = STAGE1_MODEL is not None and STAGE1_MODEL.exists()
    if USE_PRETRAINED and not have_stage1:
        problems.append(
            'THIẾU model_ucf.pth mà USE_PRETRAINED = True. Bỏ file vào một Kaggle Dataset '
            'rồi Add Input, đặt CKPT_OVERRIDE ở mục 1, hoặc đặt USE_PRETRAINED = False.')
    elif not USE_PRETRAINED and not have_stage1:
        print('Không có model_ucf.pth — không sao, train từ đầu không cần nó. '
              'Mục 8 sẽ bỏ qua dòng đối chiếu "source".')

    need = [RESCALE_DIR / n for n in
            ['ucf_train_rescale.py', 'ucf_option_rescale.py', 'ucf_eval_perclass.py',
             'losses.py', 'dataset_rescale.py', 'evaluation.py', '_bootstrap.py',
             'tests/test_losses.py']]
    # Đúng những gì _bootstrap.py cho phép src_rescale_ewc mượn từ src.
    need += [SRC_DIR / n for n in
             ['model.py', 'utils/tools.py', 'utils/layers.py',
              'utils/ucf_detectionMAP.py', 'clip/clip.py',
              'clip/bpe_simple_vocab_16e6.txt.gz']]
    need += [Path(TRAIN_LIST), Path(TEST_LIST), LIST_DIR / 'make_gt_ucf_relative.py',
             LIST_DIR / 'Temporal_Anomaly_Annotation.txt']
    for path in need:
        if not path.exists():
            problems.append(f'Thiếu file: {path}')

    # --- Bộ cờ của ucf_option_rescale.py --------------------------------------------
    if (RESCALE_DIR / 'ucf_option_rescale.py').exists():
        import ucf_option_rescale
        importlib.reload(ucf_option_rescale)
        known = {a.dest for a in ucf_option_rescale.parser._actions}
        needed = {'mu', 'rescale_mode', 'rescale_normalize', 'target_classes',
                  'regularizer', 'lambda_reg', 'lambda_auto', 'select_metric',
                  'eval_steps', 'run_tag', 'metrics_csv', 'grad_clip',
                  'use_pretrained_model', 'scheduler_milestones'}
        for name in sorted(needed - known):
            problems.append(f'ucf_option_rescale.py thiếu --{name.replace("_", "-")} — bản cũ.')

    # --- utils/layers.py đã vá chưa ---------------------------------------------------
    try:
        from utils.layers import DistanceAdj
        probe = DistanceAdj()                 # tham số nằm trên CPU
        if probe(2, 32).device.type != 'cpu':
            problems.append('utils/layers.py là BẢN CŨ: DistanceAdj ghi cứng .to("cuda").')
        del probe
    except Exception as error:
        problems.append(f'Không nạp được utils/layers.py: {error}')

    # --- Ground truth -----------------------------------------------------------------
    GT_MISSING = [LIST_DIR / n for n in ('gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy')
                  if not (LIST_DIR / n).exists()]

    # --- Feature mà hai file danh sách trỏ tới ----------------------------------------
    if FEATURE_ROOT is not None:
        for list_path in (TRAIN_LIST, TEST_LIST):
            if not Path(list_path).exists():
                continue
            rows = list(csv.DictReader(open(list_path, encoding='utf-8')))
            missing = [r for r in rows if not (Path(FEATURE_ROOT) / r['path']).exists()]
            if missing:
                problems.append(f'{Path(list_path).name}: {len(missing)}/{len(rows)} file '
                                f'.npy không tồn tại. Ví dụ: {missing[0]["path"]}')

    if problems:
        print('PREFLIGHT KHÔNG ĐẠT:')
        for problem in problems:
            print('  -', problem)
        raise RuntimeError('Sửa các mục trên rồi chạy lại cell này.')

    print()
    print('PREFLIGHT ĐẠT')
    print('  GPU          :', torch.cuda.get_device_name(0))
    print('  torch        :', torch.__version__)
    print('  FEATURE_ROOT :', FEATURE_ROOT)
    print('  Khởi tạo     :', 'từ đầu, chỉ trọng số CLIP' if not USE_PRETRAINED
          else f'nạp {STAGE1_MODEL}')
    print('  Bản code     : đủ bộ cờ rescale')
    print('  layers.py    : bản đã vá')
    if GT_MISSING:
        print('  Thiếu ground truth (mục 4.1 sẽ sinh lại):')
        for path in GT_MISSING:
            print('     ', path)
    else:
        gt = np.load(LIST_DIR / 'gt_ucf.npy')
        print('  gt_ucf.npy   :', len(gt), 'frame |', int(gt.sum()), 'frame bất thường')
    free = shutil.disk_usage(WORK).free / 1e9
    print(f'  /kaggle/working còn trống: {free:.1f} GB')

    _preflight_done = True
    return True


preflight()



### 4.1. Sinh Lại Ground Truth (chỉ khi mục 4 báo thiếu)

Ba file `gt_*.npy` sinh từ file nhãn thời gian cộng với độ dài thật của từng file đặc
trưng, nên bắt buộc phải có feature trước.


In [ ]:

if GT_MISSING:
    subprocess.run([str(x) for x in PY + [
        str(LIST_DIR / 'make_gt_ucf_relative.py'),
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        # Bắt buộc: mặc định của script tính theo thư mục hiện hành, mà thư mục hiện
        # hành là src_rescale_ewc/ -> không có file annotation ở đó.
        '--annotation', str(LIST_DIR / 'Temporal_Anomaly_Annotation.txt'),
        '--output-dir', LIST_DIR,
    ]], check=True)
    preflight(force=True)
else:
    print('Đã có đủ ground truth, bỏ qua.')



## 5. Hàm Chạy Lệnh

`build_train_cmd` giữ nguyên phần khuếch đại của bản Colab (`--mu`, `--rescale-mode`,
`--target-classes`, `--eval-steps`, `--select-metric`), và thay phần khởi tạo cùng lịch
huấn luyện bằng lịch từ-đầu của baseline.

Ba cờ truyền tường minh ở đây mà bản Colab không truyền, vì mặc định của chúng là mặc
định của giai đoạn 2 chứ không phải của baseline:

| Cờ | Bản Colab (mặc định stage 2) | Ở đây |
|---|---|---|
| `--use-pretrained-model` | `true` | **`false`** |
| `--scheduler-milestones` | `2` | **`4 8`** |
| `--grad-clip` | `1.0` | **`0`** (tắt) |

`--rescale-normalize none` và `--target-oversample 1.0` vẫn để mặc định như bản Colab.

Khác bản Colab đúng hai chỗ, cả hai đều là hệ quả của Kaggle:

- **Đường dẫn tuyệt đối**, vì `/kaggle/input` không nằm cạnh `src_rescale_ewc/` như Drive.
- **File trung gian sang `/kaggle/temp`**: `model_cur`, checkpoint chọn-theo-AUC, và
  epoch checkpoint. Bản Colab cũng đã đẩy hai cái đầu sang `/content` vì đúng lý do đó.
  Chỉ trọng số cuối của bốn lần chạy ở lại Output.


In [ ]:

import time


def run_command(cmd, log_name=None):
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    captured = []
    for line in process.stdout:
        print(line, end='')
        captured.append(line)
    process.wait()
    output = ''.join(captured)
    if log_name:
        (LOG_DIR / log_name).write_text(output, encoding='utf-8')
        print('Log saved:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return output


def build_train_cmd(tag, rescale_mode, mu, seed):
    """Phần khuếch đại giữ nguyên của bản Colab; phần lịch là lịch từ-đầu của baseline."""
    return PY + [
        'ucf_train_rescale.py',
        '--use-pretrained-model', str(USE_PRETRAINED).lower(),
        '--pretrained-model-path', STAGE1_MODEL or '',
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--target-classes', *TARGET_CLASSES,
        '--seed', seed,
        '--rescale-mode', rescale_mode,
        '--mu', mu,
        '--regularizer', REGULARIZER,
        '--lambda-reg', LAMBDA_REG,
        '--lambda-auto', LAMBDA_AUTO,
        '--max-epoch', MAX_EPOCH,
        '--lr', LR,
        '--scheduler-milestones', *SCHEDULER_MILESTONES,
        '--scheduler-rate', SCHEDULER_RATE,
        '--grad-clip', GRAD_CLIP,
        '--batch-size', BATCH_SIZE,
        '--num-workers', NUM_WORKERS,
        '--pin-memory', 'true',
        '--eval-steps', EVAL_STEPS,
        '--select-metric', SELECT_METRIC,
        '--run-tag', tag,
        '--metrics-csv', METRICS_CSV,
        '--output-model-path', MODEL_DIR / f's2_{tag}.pth',
        # Ba file dưới là trung gian, sang /kaggle/temp.
        '--checkpoint-path',      SCRATCH / f'checkpoint_s2_{tag}.pth',
        '--save-cur-path',        SCRATCH / f'model_cur_{tag}.pth',
        '--epoch-checkpoint-dir', SCRATCH / f'epoch_{tag}',
    ]


def train_1epoch(tag, rescale_mode, mu, seed):
    """Bỏ qua lần chạy đã có kết quả: chạy lại cell là đi tiếp, không phải làm lại."""
    preflight()
    output_path = MODEL_DIR / f's2_{tag}.pth'
    if output_path.exists():
        print(f'[bỏ qua] {output_path} đã tồn tại. Xoá file nếu muốn chạy lại.')
        return None
    started = time.time()
    output = run_command(build_train_cmd(tag, rescale_mode, mu, seed),
                         log_name=f'train_{tag}.log')
    print(f'Xong sau {(time.time() - started) / 3600:.2f} giờ')
    return output


print('Sẵn sàng:', [r[0] for r in RUNS if not ONLY_TAGS or r[0] in ONLY_TAGS])



## 6. Unit Test

`tests/test_losses.py` kiểm chính phần khuếch đại: trọng số theo video của Eq. (1), hệ số
gradient theo lớp của Eq. (10)-(11), và điều kiện μ = 1 phải cho kết quả trùng khít với
`rescale_mode='off'`. Chạy trên CPU, vài giây.


In [ ]:

run_command(PY + ['tests/test_losses.py'], log_name='test_losses.log')



## 7. Bốn Lần Chạy

Mỗi lần **10 epoch**: 1.250 bước tối ưu, 130 lần chấm điểm trên 290 video test.

Trong log, để ý các dòng `new best classifier_auc ... -> ...`. Dòng cuối cùng như vậy
chính là bộ trọng số được giữ, và số bước lúc đó cho biết đỉnh nằm ở đâu.

Đối chứng chạy trước, cố ý: nếu `ctrl_scratch` không rơi vào khoảng hợp lý cho một lần
chạy từ đầu (85,8–88,1 theo các lần chạy trước trong dự án) thì có gì đó sai ở setup, và
không cần tốn thêm ba lần chạy nữa mới biết.

**Bốn lần chạy không vừa một phiên 12 giờ.** Cell bỏ qua lần chạy đã có trọng số, nên bật
Persistence rồi chạy lại notebook ở phiên sau là nó đi tiếp. Muốn ép chỉ chạy vài tag thì
điền `ONLY_TAGS` ở mục 1.


In [ ]:

for tag, mode, mu, seed in RUNS:
    if ONLY_TAGS and tag not in ONLY_TAGS:
        print(f'[bỏ qua] {tag} không nằm trong ONLY_TAGS')
        continue
    print('#' * 90)
    init = 'từ đầu (chỉ CLIP)' if not USE_PRETRAINED else 'nạp model_ucf.pth'
    print(f'{tag}  |  mode={mode}  mu={mu}  seed={seed}  |  '
          f'{MAX_EPOCH} epoch, lr {LR}, {init}')
    print('#' * 90)
    train_1epoch(tag, mode, mu, seed)

done = [t for t, *_ in RUNS if (MODEL_DIR / f's2_{t}.pth').exists()]
print()
print('Đã có trọng số:', done)
print('Còn thiếu     :', [t for t, *_ in RUNS if t not in done])



## 8. Chấm Điểm Theo Lớp

Chấm mọi mô hình đã có trọng số. Nếu có `model_ucf.pth` thì chấm kèm nó dưới tên
`source` làm mốc tham chiếu (88,02).

Lưu ý cách đọc: `source` **không** còn là "điểm xuất phát" như ở bản Colab. Ở đây không
lần chạy nào bắt đầu từ nó — nó chỉ là con số của mô hình công bố, để biết một lần chạy
từ đầu đứng ở đâu so với mô hình gốc. Mốc để đánh giá phương pháp vẫn là `ctrl_scratch`
cùng seed.


In [ ]:

# Tên trước dấu '=' thành tên run trong CSV tổng hợp. Giữ đúng tên của bản Colab
# để hai bảng đặt cạnh nhau được.
# Tên trước dấu '=' thành tên run trong CSV tổng hợp.
model_specs = [f'source={STAGE1_MODEL}'] if STAGE1_MODEL else []
model_specs += [f'{tag}=' + str(MODEL_DIR / f's2_{tag}.pth') for tag, *_ in RUNS]

dropped = [s for s in model_specs if not Path(s.split('=', 1)[1]).exists()]
model_specs = [s for s in model_specs if Path(s.split('=', 1)[1]).exists()]
for spec in dropped:
    print('BỊ LOẠI (không tìm thấy file):', spec)
print('Sẽ chấm điểm', len(model_specs), 'mô hình:', [s.split('=')[0] for s in model_specs])

run_command(PY + [
    'ucf_eval_perclass.py',
    '--feature-root', FEATURE_ROOT,
    '--test-list', TEST_LIST,
    *GT_ARGS,
    '--target-classes', *TARGET_CLASSES,
    '--eval-model-paths', *model_specs,
    '--eval-output', str(PERCLASS_CSV),
], log_name='eval_perclass.log')



## 9. Bảng Kết Quả

Ba bảng:

1. **Đường cong chấm điểm** của từng lần chạy — đỉnh, điểm cuối, đỉnh nằm ở epoch nào.
2. **Hiệu giữa `class_mu12_scratch` và `ctrl_scratch` cùng seed** — đây là con số chính.
3. **Giá trị tuyệt đối theo lớp**, từ `ucf_eval_perclass.py`.

Số của bản Colab in ra chỉ để tham chiếu, và được đánh dấu rõ là **cấu hình khác**. Đừng
đặt chúng cạnh nhau như hai lần chạy so được với nhau: bên kia bắt đầu từ một mô hình đã ở
88,02, bên này bắt đầu từ số ngẫu nhiên.


In [ ]:

import pandas as pd

# Giai đoạn 2 (Colab): tinh chỉnh 1 epoch từ model_ucf.pth. CHỈ để tham chiếu, KHÔNG
# so trực tiếp được với các lần chạy từ đầu ở đây.
STAGE2_COLAB = {'ctrl_1ep': 87.81, 'class_mu12_1ep': 88.19,
                'ctrl_1ep_s1234': 87.87, 'class_mu12_1ep_s1234': 88.01}
SOURCE_AUC = 88.02

# --- 1. Đường cong ------------------------------------------------------------------
frame = pd.read_csv(METRICS_CSV)
frame = frame[frame.run != 'run']                      # dòng tiêu đề bị lặp
for column in ('epoch', 'step', 'classifier_auc'):
    frame[column] = pd.to_numeric(frame[column], errors='coerce')
# Chạy lại một tag thì CSV được NỐI THÊM chứ không ghi đè: giữ lần ghi cuối.
frame = frame.drop_duplicates(subset=['run', 'epoch', 'step'], keep='last')

print('=== ĐƯỜNG CONG CHẤM ĐIỂM (train từ đầu, 10 epoch) ===')
peaks = {}
rows = []
for tag, mode, mu, seed in RUNS:
    group = frame[frame.run == tag].sort_values(['epoch', 'step'])
    if group.empty:
        print(f'  {tag:<26} (chưa chạy)')
        continue
    best = group.loc[group.classifier_auc.idxmax()]
    peaks[tag] = float(best.classifier_auc)
    rows.append({'run': tag, 'mu': mu, 'seed': seed,
                 'so_lan_cham': len(group),
                 'dinh': round(float(best.classifier_auc), 2),
                 'dinh_o_epoch': int(best.epoch),
                 'cuoi': round(float(group.iloc[-1].classifier_auc), 2)})
if rows:
    print(pd.DataFrame(rows).set_index('run').to_string())

# --- 2. Con số chính: hiệu so với đối chứng cùng seed ---------------------------------
print()
print('=' * 78)
print('CON SỐ CHÍNH — hiệu so với đối chứng cùng seed, cùng lịch, cùng luật chọn')
print('=' * 78)
for run, base, seed in [('class_mu12_scratch', 'ctrl_scratch', 234),
                        ('class_mu12_scratch_s1234', 'ctrl_scratch_s1234', 1234)]:
    if run in peaks and base in peaks:
        delta = peaks[run] - peaks[base]
        print(f'  seed {seed:<5} {peaks[run]:.2f} (mu=12)  −  {peaks[base]:.2f} (mu=1)'
              f'   =  {delta:+.2f} điểm AUC')
    else:
        print(f'  seed {seed:<5} chưa đủ cặp')
print()
print('  Tham chiếu, KHÔNG so trực tiếp được (giai đoạn 2, tinh chỉnh từ model_ucf.pth):')
print(f"    seed 234  : {STAGE2_COLAB['class_mu12_1ep']:.2f} − "
      f"{STAGE2_COLAB['ctrl_1ep']:.2f} = +0,38")
print(f"    seed 1234 : {STAGE2_COLAB['class_mu12_1ep_s1234']:.2f} − "
      f"{STAGE2_COLAB['ctrl_1ep_s1234']:.2f} = +0,14")
print(f'    mô hình công bố source = {SOURCE_AUC}')
print()
print('  Nhắc: vòng trước đo được 0,58 AUC giữa hai lần chạy giống hệt nhau về mặt')
print('  toán học. Hiệu nhỏ hơn mức đó là nhiễu, không phải kết quả.')

# --- 3. Bảng theo lớp ----------------------------------------------------------------
summary_path = Path(str(PERCLASS_CSV).replace('.csv', '_summary.csv'))
if not summary_path.exists():
    print()
    print('Chưa có bảng theo lớp. Chạy mục 8 trước.')
else:
    summary = pd.read_csv(summary_path).set_index('run')
    cols = [c for c in ['classifier_auc', 'target_auc_c', 'target_ap_c', 'rest_auc_c',
                        'avg_mAP', 'normal_fpr@0.5'] if c in summary.columns]
    print()
    print('=== GIÁ TRỊ TUYỆT ĐỐI THEO LỚP ===')
    print(summary[cols].round(2).to_string())

    print()
    print('=== DELTA so với đối chứng CÙNG SEED ===')
    rows = []
    for run, base in [('class_mu12_scratch', 'ctrl_scratch'),
                      ('class_mu12_scratch_s1234', 'ctrl_scratch_s1234')]:
        if run in summary.index and base in summary.index:
            rows.append({'run': run,
                         **{c: round(summary.loc[run, c] - summary.loc[base, c], 2)
                            for c in cols}})
    print(pd.DataFrame(rows).set_index('run').to_string() if rows else '(chưa đủ dữ liệu)')

    summary.to_csv(RESULT_DIR / 'rescale_scratch_summary_kaggle.csv')
    print()
    print('Saved:', RESULT_DIR / 'rescale_scratch_summary_kaggle.csv')



## 10. Gom Sản Phẩm

Mọi thứ trong `/kaggle/working` tự thành Output của notebook. Tải về bằng tab **Output**
ở panel bên phải, hoặc **Save Version** để giữ vĩnh viễn.


In [ ]:

total = 0
print('Nội dung /kaggle/working:')
for path in sorted(WORK.rglob('*')):
    if path.is_file():
        size = path.stat().st_size
        total += size
        if size > 1e6:
            print(f'  {size / 1e6:8.0f} MB  {path.relative_to(WORK)}')
print()
print(f'Tổng: {total / 1e9:.2f} GB / 20 GB')

print()
print('File nhỏ (log, csv):')
for path in sorted(RESULT_DIR.rglob('*')):
    if path.is_file() and path.stat().st_size <= 1e6:
        print(f'  {path.stat().st_size / 1e3:8.0f} KB  {path.relative_to(WORK)}')



## Ghi Chú

**Cấu hình này khác cấu hình đã cho 88,19 ở bốn chỗ**, và ba trong bốn là bắt buộc phải
đổi khi train từ đầu:

| | 88,19 | Ở đây | Vì sao |
|---|---|---|---|
| Khởi tạo | `model_ucf.pth` | chỉ CLIP | yêu cầu của bài |
| Epoch | 1 | 10 | 1 epoch từ số ngẫu nhiên không hội tụ |
| lr | 2e-6 | 2e-5 | 2e-6 từ đầu quá nhỏ để học |
| grad clip | 1,0 | tắt | theo lịch baseline |

Phần khuếch đại — `mu 12`, `rescale-mode class`, bốn lớp đích, nhịp chấm 13 lần/epoch,
luật chọn `classifier_auc` — giữ nguyên hoàn toàn.

**Vì sao 88,19 không phải mục tiêu.** Nó là kết quả của việc đẩy một mô hình đã ở 88,02
lên thêm 0,17. Bỏ điểm xuất phát đó đi thì không còn lý do gì để con số cuối rơi vào cùng
chỗ. Thước đo hợp lệ duy nhất ở đây là hiệu giữa `class_mu12_scratch` và `ctrl_scratch`.

**Thay đổi trong mã nguồn.** `ucf_train_rescale.py` nhận thêm cờ
`--use-pretrained-model`, mặc định `true` nên mọi lần chạy giai đoạn 2 trước đây vẫn tái
lập nguyên vẹn. Đặt `false` thì script bỏ qua bước nạp trọng số và in
"Training VadCLIP-specific layers from scratch (CLIP weights only)". Nó cũng **từ chối
chạy** nếu bạn bật `--regularizer ewc` hoặc `l2` cùng lúc — không có θ' thì không có gì
để neo về.

**Vì sao cần cả `src` lẫn `src_rescale_ewc`.** `_bootstrap.py` nối `../src` vào
`sys.path` để dùng chung `model.py`, `clip/` và `utils/` thay vì copy. Hệ quả cho Kaggle:
hai thư mục phải nằm cạnh nhau, cùng một thư mục cha.

**Con số này được chọn thế nào.** Trọng số giữ lại là cực đại của 130 lần chấm **trên tập
kiểm tra**. Đây là cách làm của code VadCLIP gốc, nhưng phải nêu rõ trong báo cáo vì nó
khiến con số bị thiên lệch lên. Đối chứng cũng chạy dưới đúng luật ấy, nên phép so vẫn
công bằng.

**Hai seed, và phải báo cáo cả hai.** Ở giai đoạn 2, seed 1234 cho hiệu +0,14 so với
+0,38 của seed 234 — chênh nhau gần ba lần. Một seed không đủ để kết luận gì.

**Không có resume.** Phiên đứt giữa chừng thì lần chạy đang dở mất trắng, nhưng các lần đã
xong vẫn còn trong `/kaggle/working/models` và mục 7 sẽ bỏ qua chúng khi chạy lại.

**CSV được nối thêm, không ghi đè.** Chạy lại một tag sẽ thêm dòng mới vào
`rescale_scratch_metrics.csv`. Mục 9 khử trùng theo `(run, epoch, step)` và giữ lần ghi
cuối. Tên file cố ý khác `rescale_1epoch_metrics.csv` của giai đoạn 2 — trộn hai bộ số
vào một file là cách nhanh nhất để sau này đọc nhầm cấu hình.
